<a href="https://colab.research.google.com/github/Jmanav/djs-gdg-tasks/blob/task-2/AQI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [31]:
df = pd.read_excel('/content/AirQualityUCI.xlsx')
df.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,2004-03-10,18:00:00,2.6,1360.00,150,11.881723,1045.50,166.0,1056.25,113.0,1692.00,1267.50,13.60,48.875001,0.757754
1,2004-03-10,19:00:00,2.0,1292.25,112,9.397165,954.75,103.0,1173.75,92.0,1558.75,972.25,13.30,47.700000,0.725487
2,2004-03-10,20:00:00,2.2,1402.00,88,8.997817,939.25,131.0,1140.00,114.0,1554.50,1074.00,11.90,53.975000,0.750239
3,2004-03-10,21:00:00,2.2,1375.50,80,9.228796,948.25,172.0,1092.00,122.0,1583.75,1203.25,11.00,60.000000,0.786713
4,2004-03-10,22:00:00,1.6,1272.25,51,6.518224,835.50,131.0,1205.00,116.0,1490.00,1110.00,11.15,59.575001,0.788794


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9357 entries, 0 to 9356
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           9357 non-null   datetime64[ns]
 1   Time           9357 non-null   object        
 2   CO(GT)         9357 non-null   float64       
 3   PT08.S1(CO)    9357 non-null   float64       
 4   NMHC(GT)       9357 non-null   int64         
 5   C6H6(GT)       9357 non-null   float64       
 6   PT08.S2(NMHC)  9357 non-null   float64       
 7   NOx(GT)        9357 non-null   float64       
 8   PT08.S3(NOx)   9357 non-null   float64       
 9   NO2(GT)        9357 non-null   float64       
 10  PT08.S4(NO2)   9357 non-null   float64       
 11  PT08.S5(O3)    9357 non-null   float64       
 12  T              9357 non-null   float64       
 13  RH             9357 non-null   float64       
 14  AH             9357 non-null   float64       
dtypes: datetime64[ns](1),

In [33]:
df.shape[0]
df.shape

(9357, 15)

In [34]:
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')

df['Day'] = df['Date'].dt.day
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

print("Extracted columns:")
print(f"Day range: {df['Day'].min()} to {df['Day'].max()}")
print(f"Month range: {df['Month'].min()} to {df['Month'].max()}")
print(f"Year range: {df['Year'].min()} to {df['Year'].max()}")

df = df.drop('Date', axis=1)
df.head()

Extracted columns:
Day range: 1 to 31
Month range: 1 to 12
Year range: 2004 to 2005


,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Day,Month,Year
0,18:00:00,2.6,1360.00,150,11.881723,1045.50,166.0,1056.25,113.0,1692.00,1267.50,13.60,48.875001,0.757754,10,3,2004
1,19:00:00,2.0,1292.25,112,9.397165,954.75,103.0,1173.75,92.0,1558.75,972.25,13.30,47.700000,0.725487,10,3,2004
2,20:00:00,2.2,1402.00,88,8.997817,939.25,131.0,1140.00,114.0,1554.50,1074.00,11.90,53.975000,0.750239,10,3,2004
3,21:00:00,2.2,1375.50,80,9.228796,948.25,172.0,1092.00,122.0,1583.75,1203.25,11.00,60.000000,0.786713,10,3,2004
4,22:00:00,1.6,1272.25,51,6.518224,835.50,131.0,1205.00,116.0,1490.00,1110.00,11.15,59.575001,0.788794,10,3,2004


In [35]:
# AQI Formula: AQI = [(I_high - I_low) / (C_high - C_low)] * (C - C_low) + I_low
def calculate_sub_aqi(C, breakpoints):
    """
    Calculate AQI sub-index for a pollutant using linear interpolation
    C: Measured concentration
    breakpoints: List of [C_low, C_high, I_low, I_high]
    """
    for bp in breakpoints:
        C_low, C_high, I_low, I_high = bp
        if C_low <= C <= C_high:
            AQI = ((I_high - I_low) / (C_high - C_low)) * (C - C_low) + I_low
            return round(AQI)
    # If concentration exceeds all breakpoints
    return 500

# Standard AQI Breakpoints for each pollutant
# CO (Carbon Monoxide) - in mg/m³
co_breakpoints = [
    [0.0, 4.4, 0, 50],
    [4.5, 9.4, 51, 100],
    [9.5, 12.4, 101, 150],
    [12.5, 15.4, 151, 200],
    [15.5, 30.4, 201, 300],
    [30.5, 50.4, 301, 500]
]

# NMHC (Non-Methane Hydrocarbons) - in µg/m³
nmhc_breakpoints = [
    [0, 200, 0, 50],
    [201, 500, 51, 100],
    [501, 1000, 101, 150],
    [1001, 1500, 151, 200],
    [1501, 2000, 201, 300],
    [2001, 3000, 301, 500]
]

# C6H6 (Benzene) - in µg/m³
c6h6_breakpoints = [
    [0, 5, 0, 50],
    [5.1, 10, 51, 100],
    [10.1, 15, 101, 150],
    [15.1, 20, 151, 200],
    [20.1, 30, 201, 300],
    [30.1, 63.7, 301, 500]
]

# NOx (Nitrogen Oxides) - in µg/m³
nox_breakpoints = [
    [0, 100, 0, 50],
    [101, 200, 51, 100],
    [201, 700, 101, 150],
    [701, 1200, 151, 200],
    [1201, 2000, 201, 300],
    [2001, 3000, 301, 500]
]

# NO2 (Nitrogen Dioxide) - in µg/m³
no2_breakpoints = [
    [0, 53, 0, 50],
    [54, 100, 51, 100],
    [101, 360, 101, 150],
    [361, 649, 151, 200],
    [650, 1249, 201, 300],
    [1250, 2049, 301, 500]
]

# Calculate sub-indices for each pollutant
print("Calculating sub-indices for each pollutant...")
df['CO_AQI'] = df['CO(GT)'].apply(lambda x: calculate_sub_aqi(x, co_breakpoints))
df['NMHC_AQI'] = df['NMHC(GT)'].apply(lambda x: calculate_sub_aqi(x, nmhc_breakpoints))
df['C6H6_AQI'] = df['C6H6(GT)'].apply(lambda x: calculate_sub_aqi(x, c6h6_breakpoints))
df['NOx_AQI'] = df['NOx(GT)'].apply(lambda x: calculate_sub_aqi(x, nox_breakpoints))
df['NO2_AQI'] = df['NO2(GT)'].apply(lambda x: calculate_sub_aqi(x, no2_breakpoints))

# Overall AQI is the MAXIMUM of all sub-indices
df['AQI'] = df[['CO_AQI', 'NMHC_AQI', 'C6H6_AQI', 'NOx_AQI', 'NO2_AQI']].max(axis=1)


# Remove individual AQI columns, keeping only main AQI and AQI_Category
df = df.drop(['CO_AQI', 'NMHC_AQI', 'C6H6_AQI', 'NOx_AQI', 'NO2_AQI'], axis=1)

# Display results
print(f"\nAQI Range: {df['AQI'].min()} to {df['AQI'].max()}")

df.head()

Calculating sub-indices for each pollutant...

AQI Range: 18 to 500


,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Day,Month,Year,AQI
0,18:00:00,2.6,1360.00,150,11.881723,1045.50,166.0,1056.25,113.0,1692.00,1267.50,13.60,48.875001,0.757754,10,3,2004,119
1,19:00:00,2.0,1292.25,112,9.397165,954.75,103.0,1173.75,92.0,1558.75,972.25,13.30,47.700000,0.725487,10,3,2004,94
2,20:00:00,2.2,1402.00,88,8.997817,939.25,131.0,1140.00,114.0,1554.50,1074.00,11.90,53.975000,0.750239,10,3,2004,103
3,21:00:00,2.2,1375.50,80,9.228796,948.25,172.0,1092.00,122.0,1583.75,1203.25,11.00,60.000000,0.786713,10,3,2004,105
4,22:00:00,1.6,1272.25,51,6.518224,835.50,131.0,1205.00,116.0,1490.00,1110.00,11.15,59.575001,0.788794,10,3,2004,104


In [36]:
def create_seq(data, window_size):
    X = [] #input seq
    y = [] #target
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

X,y = create_seq(df['AQI'],24)

In [37]:
X.shape

(9333, 24)

In [38]:
#we can see that its 2D and lstm req 3D input  --> (samples,timesteps,features)
X = X.reshape(X.shape[0],X.shape[1],1)

In [39]:
train_size = int(len(X) * 0.8)

X_train,X_test = X[:train_size],X[train_size:]
y_train,y_test = y[:train_size],y[train_size:]

In [40]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

def build_lstm(units,alpha,window_size):
    model = Sequential()
    model.add(LSTM(units, input_shape=(window_size, 1))) #units->number of neurons
                                                         #for each training example we'll give you "window_size" seq vals for 1 feature=aqi
    model.add(Dense(1)) #number of output neurons
    model.compile(loss='mean_squared_error', optimizer='adam')
    return model

we'll use pso to find the best parameters

LSTM units: 32 → 256

Learning rate: 0.0001 → 0.01

Batch size: 16 → 128

Epochs: 10 → 100

In [41]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

def evaluate_lstm(params):
    units = int(params[0])          # number of LSTM neurons
    lr = params[1]                  # learning rate
    batch_size = int(params[2])     # batch size

    model = Sequential()
    model.add(LSTM(units, input_shape=(X_train.shape[1], 1)))
    model.add(Dense(1))
    model.compile(loss='mse', optimizer=Adam(learning_rate=lr))

    history = model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=batch_size,
        verbose=0,
        validation_data=(X_test, y_test)
    )

    loss = history.history['val_loss'][-1]
    return loss   # PSO tries to minimize this


In [42]:
pip install pyswarms

In [43]:
from pyswarms.single import GlobalBestPSO

# Define bounds for hyperparameters
# [units, lr, batch_size]
lower_bounds = [10, 0.0001, 8]
upper_bounds = [100, 0.01, 64]

bounds = (lower_bounds, upper_bounds)

# Initialize PSO
optimizer = GlobalBestPSO(
    n_particles=10,
    dimensions=3,
    options={'c1': 2, 'c2': 2, 'w': 0.5},
    bounds=bounds
)

# PSO optimization
best_cost, best_pos = optimizer.optimize(
    lambda params: np.array([evaluate_lstm(p) for p in params]),
    iters=5
)

print("Best Hyperparameters Found:")
print("Units:", int(best_pos[0]))
print("Learning Rate:", best_pos[1])
print("Batch Size:", int(best_pos[2]))


2025-11-23 13:15:29,989 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 2, 'c2': 2, 'w': 0.5}
pyswarms.single.global_best:   0%|          |0/5/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=229
2025-11-23 13:33:52,783 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 229.02931213378906, best pos: [6.80326242e+01 9.97111838e-03 1.54254286e+01]


Best Hyperparameters Found:
Units: 68
Learning Rate: 0.0099711183803691
Batch Size: 15


In [44]:
#we'll train lstm with the best params we got using pso
best_units = int(best_pos[0])
best_lr = best_pos[1]
best_batch = int(best_pos[2])

model = Sequential()
model.add(LSTM(best_units, input_shape=(X_train.shape[1],1)))
model.add(Dense(1))
model.compile(loss='mse', optimizer=Adam(learning_rate=best_lr))

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=best_batch,
    validation_data=(X_test, y_test)
)


Epoch 1/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - loss: 170924.7344 - val_loss: 69283.0625
Epoch 2/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - loss: 47668.1797 - val_loss: 16940.1309
Epoch 3/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - loss: 13810.5713 - val_loss: 2362.0825
Epoch 4/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 4607.4053 - val_loss: 467.5927
Epoch 5/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 3032.9731 - val_loss: 165.2215
Epoch 6/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 2861.2041 - val_loss: 48.2942
Epoch 7/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 3127.5571 - val_loss: 43.7891
Epoch 8/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 2844.7852 - val_loss: 1164.1150
Epoch 9/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 3217.8848 - val_loss: 924.1995
Epoch 10/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 3674.4895 - val_loss: 1003.9752
Epoch 11/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 3698.0